# Loading the dataset

In [1]:
import pandas as pd
import numpy as np

url = (
    "https://raw.githubusercontent.com/"
    "DataTalksClub/machine-learning-zoomcamp/main/"
    "cohorts/2026/data/course_lead_scoring_2026.csv"
)

df = pd.read_csv(url)

print(df.shape)
df.head()

(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [2]:
df.isnull().sum() # seeing how many missing values are in each column

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [3]:
df.dtypes # checking the data types of each column

lead_source                  object
industry                     object
employment_status            object
location                     object
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

# Question 1

In [4]:
# selecting categorical and numerical columns 
categorical_cols = df.select_dtypes(include="object").columns # selecting categorical columns
numerical_cols = df.select_dtypes(include=np.number).columns # selecting numerical columns

df[categorical_cols] = df[categorical_cols].fillna("NA") # filling missing values in categorical columns with "NA"
df[numerical_cols] = df[numerical_cols].fillna(0.0) # filling missing values in numerical columns with 0.0

df.isnull().sum().sum() # checking if there are any missing values left in the dataframe after filling them

np.int64(0)

In [5]:
df["industry"].mode() # checking the most common value in the "industry" column

0    technology
Name: industry, dtype: object

# Question 2

In [6]:
# calculating the correlation matrix for numerical columns
corr = df.select_dtypes(include=np.number).corr() # calculating the correlation matrix for numerical columns

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
] # defining pairs of columns to calculate correlation for

for col1, col2 in pairs:
    print(col1, "and", col2, ":", round(corr.loc[col1, col2], 4)) # printing the correlation between each pair of columns with 4 decimal places

interaction_count and lead_score : 0.9157
number_of_courses_viewed and lead_score : 0.7572
number_of_courses_viewed and interaction_count : 0.7216
annual_income and interaction_count : 0.1228


# Question 3

In [7]:
# importing necessary libraries for machine learning tasks
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import mutual_info_score

In [8]:
# splitting the dataset into training and testing sets, and then further splitting the training set into training and validation sets

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
) # splitting the dataset into training and testing sets, with 20% of the data reserved for testing

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
) # further splitting the training set into training and validation sets, with 25% of the training data reserved for validation

y_train = df_train["converted"]
y_val = df_val["converted"] # extracting the target variable "converted" from the training and validation sets

X_train = df_train.drop(columns=["converted"])
X_val = df_val.drop(columns=["converted"]) # dropping the target variable "converted" from the training and validation sets to create feature sets


In [9]:
# calculating mutual information scores for categorical columns in the training set

cat_cols = X_train.select_dtypes(include="object").columns # selecting categorical columns from the training set

mi_scores = {
    col: mutual_info_score(y_train, X_train[col])
    for col in cat_cols
} # calculating mutual information scores for each categorical column in the training set with respect to the target variable "converted"

print("Q3 scores:", {
    col: round(score, 2)
    for col, score in mi_scores.items()
})
print("Q3 answer:", max(mi_scores, key=mi_scores.get)) # printing the mutual information scores rounded to 2 decimal places and the column with the highest mutual information score

Q3 scores: {'lead_source': 0.03, 'industry': 0.0, 'employment_status': 0.02, 'location': 0.0}
Q3 answer: lead_source


In [10]:
# defining a function to calculate accuracy of a logistic regression model with given training and validation sets, and regularization parameter C helper function to calculate accuracy of a logistic regression model with given training and validation sets, and regularization parameter C

def get_accuracy(X_tr, X_va, C=1.0):
    dv = DictVectorizer(sparse=False)

    X_tr_encoded = dv.fit_transform(
        X_tr.to_dict(orient="records")
    ) # encoding the training set using DictVectorizer to convert categorical variables into numerical format
    X_va_encoded = dv.transform(
        X_va.to_dict(orient="records")
    ) # encoding the validation set using the same DictVectorizer to ensure consistent encoding of categorical variables

    model = LogisticRegression(
        solver="liblinear",
        C=C,
        max_iter=1000,
        random_state=42
    ) # defining a logistic regression model with specified solver, regularization parameter C, maximum iterations, and random state for reproducibility

    model.fit(X_tr_encoded, y_train)
    predictions = model.predict(X_va_encoded) # making predictions on the validation set using the trained logistic regression model

    return accuracy_score(y_val, predictions)

# Question 4

In [11]:
# calculating the accuracy of the logistic regression model on the validation set using the default regularization parameter C=1.0
baseline = get_accuracy(X_train, X_val, C=1.0)
print("Q4 accuracy:", round(baseline, 2))

Q4 accuracy: 0.65


# Question 5

In [12]:
# calculating the differences in accuracy when dropping each specified column from the training and validation sets
differences = {}

for col in [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]: # iterating over the specified columns to calculate the accuracy difference when each column is dropped from the training and validation sets
    reduced_accuracy = get_accuracy(
        X_train.drop(columns=[col]),
        X_val.drop(columns=[col]),
        C=1.0
    ) # calculating the accuracy of the logistic regression model on the validation set after dropping the specified column from the training and validation sets

    differences[col] = abs(baseline - reduced_accuracy)

print("Q5 differences:", differences)
print("Q5 answer:", min(differences, key=differences.get))


Q5 differences: {'lead_source': 0.0030000000000000027, 'number_of_courses_viewed': 0.0020000000000000018, 'interaction_count': 0.04400000000000004}
Q5 answer: number_of_courses_viewed


# Question 6

In [13]:
# testing different values of the regularization parameter C to find the best one based on accuracy on the validation set
c_values = [0.000001, 0.00001, 0.0001, 0.001]

c_scores = {
    C: get_accuracy(X_train, X_val, C=C)
    for C in c_values
} # calculating the accuracy of the logistic regression model on the validation set for each value of the regularization parameter C

print("Q6 accuracies:", c_scores)

best_C = max(c_values, key=lambda C: (c_scores[C], -C))
print("Q6 answer:", best_C)

Q6 accuracies: {1e-06: 0.598, 1e-05: 0.598, 0.0001: 0.613, 0.001: 0.645}
Q6 answer: 0.001
